In [3]:
import os
import re
import sys
import contextlib
from pathlib import Path
from collections import defaultdict

# Numerical Computing & Data Handling:
import numpy as np
import pandas as pd

# Scientific Computing & Statistics:
from scipy.io import loadmat, savemat
from scipy.stats import ttest_ind
from statsmodels.stats.multitest import multipletests

# Neuroimaging:
import nibabel as nib
from nilearn.datasets import fetch_atlas_schaefer_2018
from nilearn.maskers import NiftiLabelsMasker
from neuromaps.datasets import fetch_fslr

# Network Analysis:
import networkx as nx
from networkx.algorithms import community

# Visualization:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors
import matplotlib.patches as patches
from matplotlib.lines import Line2D
import seaborn as sns

# Brain Surface Visualization:
from surfplot import Plot

# Utilities:
from tqdm.auto import tqdm

# Warning Suppression Utility:
@contextlib.contextmanager
def suppress_vtk_warnings():
    devnull = open(os.devnull, "w")
    old_stderr = os.dup(2)
    os.dup2(devnull.fileno(), 2)
    try:
        yield
    finally:
        os.dup2(old_stderr, 2)
        os.close(old_stderr)
        devnull.close()

In [4]:
dataset_root = Path("../../dataset/derivatives")

yeo7_atlas_path = Path("Atlas/Yeo7NetworksAtlas.nii.gz")
yeo7_atlas_img = nib.load(str(yeo7_atlas_path))

output_root = Path("outputs_yeo7_activation_all_methods")
output_root.mkdir(exist_ok=True, parents=True)

network_names = [
    "Visual",
    "Somatomotor",
    "Dorsal_Attention",
    "Ventral_Attention",
    "Limbic",
    "Frontoparietal",
    "Default_Mode"
]

# Thresholds to test
z_thresholds = [1.5, 2.0, 2.25]
percentile_thresholds = [0.90, 0.95, 0.97]

# ============================================================
# LOAD METADATA AND FILTER SUBJECTS/SESSIONS BEFORE PROCESSING
# ============================================================

metadata_path = Path("OASIS3_metadata_clean.csv")  
# If the CSV is in the same folder as the notebook, use:
# metadata_path = Path("OASIS3_metadata_clean.csv")

metadata = pd.read_csv(metadata_path)

print("Metadata columns:")
print(metadata.columns)
print(metadata.head())

# Make sure IDs are strings
metadata["Subject_ID"] = metadata["Subject_ID"].astype(str)
metadata["Session_ID"] = metadata["Session_ID"].astype(str)

# Convert metadata IDs to folder-style IDs
# Example:
# OAS30001 -> sub-OAS30001
# d0129    -> ses-d0129
metadata["subject_folder"] = "sub-" + metadata["Subject_ID"]
metadata["session_folder"] = "ses-" + metadata["Session_ID"]

valid_pairs = set(zip(metadata["subject_folder"], metadata["session_folder"]))

print("Valid subject-session pairs in CSV:", len(valid_pairs))


# ============================================================
# SEARCH ALL FUNCTIONAL FILES
# ============================================================

all_func_files = sorted(dataset_root.glob(
    "sub-*/ses-*/func/cleaned/*desc-cleaned_scrubbed_gsr_bold.nii.gz"
))

print("Total functional files found before filtering:", len(all_func_files))


# ============================================================
# FILTER FUNCTIONAL FILES USING METADATA CSV
# ============================================================

func_files = []

for func_file in all_func_files:
    subject = next((p for p in func_file.parts if p.startswith("sub-")), None)
    session = next((p for p in func_file.parts if p.startswith("ses-")), None)

    if subject is None or session is None:
        continue

    if (subject, session) in valid_pairs:
        func_files.append(func_file)

print("Functional files after CSV filtering:", len(func_files))

print("\nFirst files that will be processed:")
for f in func_files[:10]:
    print(f)

Metadata columns:
Index(['Subject_ID', 'Session_ID', 'DEMENTED', 'NORMCOG', 'Age'], dtype='str')
  Subject_ID Session_ID  DEMENTED  NORMCOG    Age
0   OAS30001      d0129         0        1  65.54
1   OAS30002      d0653         0        1  69.04
2   OAS30003      d0558         0        1  60.34
3   OAS30004      d1101         0        1  58.14
4   OAS30005      d0581         0        1  49.65
Valid subject-session pairs in CSV: 1086
Total functional files found before filtering: 1188
Functional files after CSV filtering: 1086

First files that will be processed:
../../dataset/derivatives/sub-OAS30001/ses-d0129/func/cleaned/sub-OAS30001_ses-d0129_task-rest_run-02_space-MNI152NLin2009cAsym_res-2_desc-cleaned_scrubbed_gsr_bold.nii.gz
../../dataset/derivatives/sub-OAS30002/ses-d0653/func/cleaned/sub-OAS30002_ses-d0653_task-rest_run-02_space-MNI152NLin2009cAsym_res-2_desc-cleaned_scrubbed_gsr_bold.nii.gz
../../dataset/derivatives/sub-OAS30003/ses-d0558/func/cleaned/sub-OAS30003_ses-d0558_t

In [5]:
masker = NiftiLabelsMasker(
    labels_img=yeo7_atlas_img,
    standardize=False,
    detrend=False,
    verbose=0
)


def summarize_binary_activation(binary_ts, method_name):
    """
    Computes summary metrics for a binary activation time series.
    
    Parameters
    ----------
    binary_ts : pd.DataFrame
        Binary time series with shape timepoints x networks.
    method_name : str
        Name of the thresholding method, e.g. 'Z>1.5' or 'P90'.
    
    Returns
    -------
    summary : dict
        Dictionary containing activation summary metrics.
    """

    activation_rate_per_network = binary_ts.mean() * 100
    no_active_timepoints = (binary_ts.sum(axis=1) == 0).mean() * 100
    mean_active_networks_per_TR = binary_ts.sum(axis=1).mean()
    states = binary_ts.astype(str).agg("".join, axis=1)
    n_unique_states = states.nunique()

    summary = {
        "method": method_name,
        "mean_activation_%": activation_rate_per_network.mean(),
        "min_activation_%": activation_rate_per_network.min(),
        "max_activation_%": activation_rate_per_network.max(),
        "no_active_timepoints_%": no_active_timepoints,
        "mean_active_networks_per_TR": mean_active_networks_per_TR,
        "n_unique_states": n_unique_states
    }

    for network in binary_ts.columns:
        summary[f"activation_%_{network}"] = activation_rate_per_network[network]

    return summary


def process_one_functional_file_all_methods(
    func_file,
    masker,
    output_root,
    network_names,
    z_thresholds=[1.5, 2.0, 2.25],
    percentile_thresholds=[0.90, 0.95, 0.97]
):
    """
    Processes one resting-state fMRI file:
    - extracts mean Yeo-7 network time series
    - computes z-scored time series
    - generates binary activation time series using z-score thresholds
    - generates binary activation time series using percentile thresholds
    - saves all outputs as CSV files
    """

    func_file = Path(func_file)

    # Extract subject, session, and run identifiers from the file path/name
    subject = [p for p in func_file.parts if p.startswith("sub-")][0]
    session = [p for p in func_file.parts if p.startswith("ses-")][0]

    run_parts = [p for p in func_file.name.split("_") if p.startswith("run-")]
    run = run_parts[0] if len(run_parts) > 0 else "run-unknown"

    base_name = f"{subject}_{session}_{run}"

    out_dir = output_root / subject / session
    out_dir.mkdir(exist_ok=True, parents=True)

    # --------------------------------------------------------
    # 1. Extract continuous mean time series
    # --------------------------------------------------------
    time_series = masker.fit_transform(str(func_file))

    if time_series.shape[1] != len(network_names):
        raise ValueError(
            f"Expected {len(network_names)} networks, but obtained "
            f"{time_series.shape[1]} in {func_file}"
        )

    df_ts = pd.DataFrame(time_series, columns=network_names)

    # --------------------------------------------------------
    # 2. Compute z-score per network
    # --------------------------------------------------------
    df_z = (df_ts - df_ts.mean()) / df_ts.std(ddof=0)
    df_z = df_z.replace([np.inf, -np.inf], np.nan).fillna(0)

    # --------------------------------------------------------
    # 3. Save continuous and z-scored time series
    # --------------------------------------------------------
    ts_path = out_dir / f"{base_name}_yeo7_timeseries.csv"
    z_path = out_dir / f"{base_name}_yeo7_zscore.csv"

    df_ts.to_csv(ts_path, index=False)
    df_z.to_csv(z_path, index=False)

    summaries = []

    common_info = {
        "subject": subject,
        "session": session,
        "run": run,
        "func_file": str(func_file),
        "n_timepoints": df_ts.shape[0],
        "n_networks": df_ts.shape[1],
        "timeseries_file": str(ts_path),
        "zscore_file": str(z_path)
    }

    # --------------------------------------------------------
    # 4. Binary activation using z-score thresholds
    # --------------------------------------------------------
    for z_thr in z_thresholds:
        method_name = f"Z>{z_thr}"
        method_file_label = f"Z{str(z_thr).replace('.', '')}"

        binary_z = (df_z > z_thr).astype(int)

        bin_path = out_dir / f"{base_name}_yeo7_binary_{method_file_label}.csv"
        binary_z.to_csv(bin_path, index=False)

        summary = summarize_binary_activation(binary_z, method_name)
        summary.update(common_info)
        summary["binary_file"] = str(bin_path)
        summary["threshold_type"] = "zscore"
        summary["threshold_value"] = z_thr

        summaries.append(summary)

    # --------------------------------------------------------
    # 5. Binary activation using percentile thresholds
    # --------------------------------------------------------
    for p in percentile_thresholds:
        method_name = f"P{int(p * 100)}"
        method_file_label = f"P{int(p * 100)}"

        thresholds = df_ts.quantile(p)
        binary_p = (df_ts > thresholds).astype(int)

        bin_path = out_dir / f"{base_name}_yeo7_binary_{method_file_label}.csv"
        binary_p.to_csv(bin_path, index=False)

        summary = summarize_binary_activation(binary_p, method_name)
        summary.update(common_info)
        summary["binary_file"] = str(bin_path)
        summary["threshold_type"] = "percentile"
        summary["threshold_value"] = p

        summaries.append(summary)

    return summaries

In [6]:

all_summaries = []
errors = []

for func_file in tqdm(func_files):
    try:
        summaries = process_one_functional_file_all_methods(
            func_file=func_file,
            masker=masker,
            output_root=output_root,
            network_names=network_names,
            z_thresholds=z_thresholds,
            percentile_thresholds=percentile_thresholds
        )

        all_summaries.extend(summaries)

    except Exception as e:
        errors.append({
            "func_file": str(func_file),
            "error": str(e)
        })

summary_df = pd.DataFrame(all_summaries)
errors_df = pd.DataFrame(errors)

summary_path = output_root / "activation_summary_all_methods.csv"
errors_path = output_root / "activation_errors.csv"

summary_df.to_csv(summary_path, index=False)
errors_df.to_csv(errors_path, index=False)

print("Functional files processed:", len(func_files))
print("Summary rows generated:", len(summary_df))
print("Errors:", len(errors_df))

summary_df.head()

  0%|          | 0/1086 [00:00<?, ?it/s]

Functional files processed: 1086
Summary rows generated: 6516
Errors: 0


,method,mean_activation_%,min_activation_%,max_activation_%,no_active_timepoints_%,mean_active_networks_per_TR,n_unique_states,activation_%_Visual,activation_%_Somatomotor,activation_%_Dorsal_Attention,...,session,run,func_file,n_timepoints,n_networks,timeseries_file,zscore_file,binary_file,threshold_type,threshold_value
0,Z>1.5,7.660455,6.521739,10.144928,66.666667,0.536232,15,6.521739,7.971014,10.144928,...,ses-d0129,run-02,../../dataset/derivatives/sub-OAS30001/ses-d01...,138,7,outputs_yeo7_activation_all_methods/sub-OAS300...,outputs_yeo7_activation_all_methods/sub-OAS300...,outputs_yeo7_activation_all_methods/sub-OAS300...,zscore,1.50
1,Z>2.0,3.416149,0.724638,5.797101,81.884058,0.239130,12,3.623188,5.797101,0.724638,...,ses-d0129,run-02,../../dataset/derivatives/sub-OAS30001/ses-d01...,138,7,outputs_yeo7_activation_all_methods/sub-OAS300...,outputs_yeo7_activation_all_methods/sub-OAS300...,outputs_yeo7_activation_all_methods/sub-OAS300...,zscore,2.00
2,Z>2.25,2.277433,0.724638,4.347826,88.405797,0.159420,9,2.898551,4.347826,0.724638,...,ses-d0129,run-02,../../dataset/derivatives/sub-OAS30001/ses-d01...,138,7,outputs_yeo7_activation_all_methods/sub-OAS300...,outputs_yeo7_activation_all_methods/sub-OAS300...,outputs_yeo7_activation_all_methods/sub-OAS300...,zscore,2.25
3,P90,10.144928,10.144928,10.144928,56.521739,0.710145,20,10.144928,10.144928,10.144928,...,ses-d0129,run-02,../../dataset/derivatives/sub-OAS30001/ses-d01...,138,7,outputs_yeo7_activation_all_methods/sub-OAS300...,outputs_yeo7_activation_all_methods/sub-OAS300...,outputs_yeo7_activation_all_methods/sub-OAS300...,percentile,0.90
4,P95,5.072464,5.072464,5.072464,74.637681,0.355072,14,5.072464,5.072464,5.072464,...,ses-d0129,run-02,../../dataset/derivatives/sub-OAS30001/ses-d01...,138,7,outputs_yeo7_activation_all_methods/sub-OAS300...,outputs_yeo7_activation_all_methods/sub-OAS300...,outputs_yeo7_activation_all_methods/sub-OAS300...,percentile,0.95


In [7]:
method_summary_mean = summary_df.groupby("method")[
    [
        "mean_activation_%",
        "no_active_timepoints_%",
        "mean_active_networks_per_TR",
        "n_unique_states"
    ]
].mean()

method_summary_std = summary_df.groupby("method")[
    [
        "mean_activation_%",
        "no_active_timepoints_%",
        "mean_active_networks_per_TR",
        "n_unique_states"
    ]
].std()

display(method_summary_mean)
display(method_summary_std)

,mean_activation_%,no_active_timepoints_%,mean_active_networks_per_TR,n_unique_states
method,,,,
P90,10.369990,53.106664,0.725899,20.588398
P95,5.431635,72.203481,0.380214,14.029466
P97,3.482812,81.190830,0.243797,11.291897
Z>1.5,6.684928,66.896149,0.467945,15.333333
Z>2.0,2.458567,86.303024,0.172100,8.894107
Z>2.25,1.441567,91.666205,0.100910,6.719153


,mean_activation_%,no_active_timepoints_%,mean_active_networks_per_TR,n_unique_states
method,,,,
P90,1.261167,4.729337,0.088282,3.304935
P95,1.405345,3.525577,0.098374,2.222436
P97,1.464732,3.011714,0.102531,1.719626
Z>1.5,0.833729,5.208466,0.058361,2.936083
Z>2.0,0.616916,3.342026,0.043184,2.120521
Z>2.25,0.506697,2.662318,0.035469,1.895749


In [8]:
method_summary = summary_df.groupby("method")[
    [
        "mean_activation_%",
        "no_active_timepoints_%",
        "mean_active_networks_per_TR",
        "n_unique_states"
    ]
].mean().reset_index()

candidates = method_summary[
    (method_summary["mean_activation_%"] >= 5) &
    (method_summary["mean_activation_%"] <= 12) &
    (method_summary["no_active_timepoints_%"] <= 80) &
    (method_summary["n_unique_states"] >= 10)
]

candidates

,method,mean_activation_%,no_active_timepoints_%,mean_active_networks_per_TR,n_unique_states
0,P90,10.369990,53.106664,0.725899,20.588398
1,P95,5.431635,72.203481,0.380214,14.029466
3,Z>1.5,6.684928,66.896149,0.467945,15.333333
